**Import the Libraries**

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [2]:
base_url = 'https://books.toscrape.com/'

res = requests.get(base_url)
res.status_code

200

In [3]:
soup = BeautifulSoup(res.text, "html.parser")
soup

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

**Extract Book Details for Page 1**

In [ ]:
books = soup.find_all('h3')

start_time = time.time()
books_extracted = 0

for book in books:
  book_url = book.find('a')['href']
  book_response = requests.get(base_url + book_url)
  book_soup = BeautifulSoup(book_response.content, 'html.parser')

  title = book_soup.find('h1').text.strip()
  category = book_soup.find('ul', class_ = 'breadcrumb').find_all('a')[2].text.strip()
  price = book_soup.find('p', class_ = 'price_color').text.strip()
  how_many_available = book_soup.find('p', class_ = 'instock availability').text.strip()
  rating = book_soup.find('p', class_ = 'star-rating')['class'][1]

  books_extracted += 1

  end_time = time.time()
  total_time = end_time - start_time

  print("Title: ", title)
  print("Category: ", category)
  print("Price: ", price)
  print("In Stock: ", how_many_available)
  print("Rating: ", rating)
  print("*********")

print("Total books extracted: ", books_extracted)

Title:  A Light in the Attic
Category:  Poetry
Price:  £51.77
In Stock:  In stock (22 available)
Rating:  Three
*********
Title:  Tipping the Velvet
Category:  Historical Fiction
Price:  £53.74
In Stock:  In stock (20 available)
Rating:  One
*********
Title:  Soumission
Category:  Fiction
Price:  £50.10
In Stock:  In stock (20 available)
Rating:  One
*********
Title:  Sharp Objects
Category:  Mystery
Price:  £47.82
In Stock:  In stock (20 available)
Rating:  Four
*********
Title:  Sapiens: A Brief History of Humankind
Category:  History
Price:  £54.23
In Stock:  In stock (20 available)
Rating:  Five
*********
Title:  The Requiem Red
Category:  Young Adult
Price:  £22.65
In Stock:  In stock (19 available)
Rating:  One
*********
Title:  The Dirty Little Secrets of Getting Your Dream Job
Category:  Business
Price:  £33.34
In Stock:  In stock (19 available)
Rating:  Four
*********
Title:  The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Category:  Def

**Extract Details for all 50 Pages**

In [ ]:
from urllib.parse import urljoin

books_data = []
books_extracted = 0
start_time = time.time()

for page_num in range(1, 51):
  page_url = base_url
  if page_num != 1:
    page_url = base_url + f'catalogue/page-{page_num}.html'

  print(f"Page {page_num} of 50")

  page_response = requests.get(page_url)
  page_soup = BeautifulSoup(page_response.content, 'html.parser')

  books = page_soup.find_all('h3')

  for book in books:

    book_url = urljoin(page_url, book.find('a')['href'])
    book_response = requests.get(book_url)
    book_soup = BeautifulSoup(book_response.content, 'html.parser')

    title = book_soup.find('h1').text.strip()
    category = book_soup.find('ul', class_ = 'breadcrumb')
    if category:
      category_links = category.find_all('a')
      if len(category_links) >= 3:
        category = category_links[2].text.strip()
      else:
        category = "Category not found"
    else:
      category = "Breadcrumb element not found"
    price = book_soup.find('p', class_ = 'price_color').text.strip()
    how_many_available = book_soup.find('p', class_ = 'instock availability').text.strip()
    rating = book_soup.find('p', class_ = 'star-rating')['class'][1]

    books_extracted += 1
    books_data.append([title, category, price, how_many_available, rating])

end_time = time.time()
total_time = end_time - start_time
print("Total books extracted: ", books_extracted)

Page 1 of 50
Page 2 of 50
Page 3 of 50
Page 4 of 50
Page 5 of 50
Total books extracted:  100


In [ ]:
df = pd.DataFrame(books_data, columns = ['Title', 'Category', 'Price', 'Availability', 'Rating'])
df

,Title,Category,Price,Availability,Rating
0,A Light in the Attic,Poetry,£51.77,In stock (22 available),Three
1,Tipping the Velvet,Historical Fiction,£53.74,In stock (20 available),One
2,Soumission,Fiction,£50.10,In stock (20 available),One
3,Sharp Objects,Mystery,£47.82,In stock (20 available),Four
4,Sapiens: A Brief History of Humankind,History,£54.23,In stock (20 available),Five
...,...,...,...,...,...
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Sequential Art,£19.92,In stock (16 available),Two
96,"Layered: Baking, Building, and Styling Spectac...",Food and Drink,£40.11,In stock (16 available),One
97,Judo: Seven Steps to Black Belt (an Introducto...,Add a comment,£53.90,In stock (16 available),Two
98,Join,Science Fiction,£35.67,In stock (16 available),Five
